Start by importing the required packages.

In [115]:
import pandas as pd
!pip install geopy

   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   ---------------------------------------- 125.4/125.4 kB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 40.7/40.7 kB 1.9 MB/s eta 0:00:00


Load in the CSV file and look and the column names and datatypes.

In [125]:
atp2023 = pd.read_csv('atp_matches_2023.csv')
atp2023.dtypes

tourney_id             object
tourney_name           object
surface                object
draw_size               int64
tourney_level          object
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed           float64
winner_entry           object
winner_name            object
winner_hand            object
winner_ht             float64
winner_ioc             object
winner_age            float64
loser_id                int64
loser_seed            float64
loser_entry            object
loser_name             object
loser_hand             object
loser_ht              float64
loser_ioc              object
loser_age             float64
score                  object
best_of                 int64
round                  object
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

Drop unnecessary columns and list unique values in the tourney_name column.

In [127]:
atp2023 = atp2023[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals', 'Davis Cup

Remove the values that contain "Davis Cup". These won't be used in the analysis.

In [95]:
atp2023 = atp2023[~atp2023['tourney_name'].str.contains('Davis Cup', case=False, na=False)]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=obj

Remove "Masters" from any of the values. This will help find the locations.

In [96]:
atp2023["tourney_name"] = atp2023["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Roland Garros', 's Hertogenbosch',
       'Stuttgart', 'Halle', "Queen's Club", 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos', 'Washington', 'Canada',
       'Cincinnati', 'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=object)

Change the tournament names without clear locations to their actual locations.

In [97]:
atp2023["tourney_name"] = atp2023["tourney_name"].replace({
    "Australian Open": "Melbourne",
    "Roland Garros": "Paris",
    "Queen's Club": "London",
    "Canada": "Toronto",
    "Us Open": "New York",
    "Tour Finals": "Turin",
    "NextGen Finals": "Jeddah"
})
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Melbourne', 'Cordoba', 'Dallas', 'Montpellier', 'Delray Beach',
       'Buenos Aires', 'Rotterdam', 'Doha', 'Rio De Janeiro', 'Marseille',
       'Acapulco', 'Dubai', 'Santiago', 'Indian Wells', 'Miami',
       'Estoril', 'Houston', 'Marrakech', 'Monte Carlo', 'Barcelona',
       'Munich', 'Banja Luka', 'Madrid', 'Rome', 'Geneva', 'Lyon',
       'Paris', 's Hertogenbosch', 'Stuttgart', 'Halle', 'London',
       'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad', 'Bastad',
       'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel', 'Los Cabos',
       'Washington', 'Toronto', 'Cincinnati', 'Winston-Salem', 'New York',
       'Chengdu', 'Zhuhai', 'Astana', 'Beijing', 'Shanghai', 'Stockholm',
       'Tokyo', 'Antwerp', 'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin',
       'Jeddah'], dtype=object)

Change "United Cup" to the locations where the event is held.

In [98]:
atp2023["tourney_name"] = atp2023["tourney_name"].apply(
    lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" else [x]
)

atp2023 = atp2023.explode("tourney_name")
atp2023["tourney_name"].unique()

array(['Brisbane', 'Sydney', 'Perth', 'Adelaide 1', 'Pune', 'Auckland',
       'Adelaide 2', 'Melbourne', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch', 'Stuttgart',
       'Halle', 'London', 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Toronto', 'Cincinnati',
       'Winston-Salem', 'New York', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp', 'Basel',
       'Vienna', 'Metz', 'Sofia', 'Turin', 'Jeddah'], dtype=object)

Create a dataframe that contains a list of the different locations tournaments are held and the locations for each one.

In [117]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="atp_2023_analysis")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # Nominatim rate limit

unique_cities = atp2023["tourney_name"].unique()

location_data = {}
for city in unique_cities:
    location = geocode(city)
    if location:
        location_data[city] = (location.latitude, location.longitude)
    else:
        print(f"Could not geocode: {city}")  # flag any misses

coord_df = pd.DataFrame.from_dict(
    location_data, orient="index", columns=["latitude", "longitude"]
)
coord_df.index.name = "tourney_name"
coord_df = coord_df.reset_index()

In [119]:
coord_df.to_csv("atp_city_coords.csv", index=False)

In [121]:
coord_df

,tourney_name,latitude,longitude
0,Brisbane,-27.468962,153.023501
1,Sydney,-33.869844,151.208285
2,Perth,-31.955897,115.860578
3,Adelaide 1,-34.819459,138.504749
4,Pune,18.521374,73.854507
...,...,...,...
63,Vienna,48.208354,16.372504
64,Metz,49.119696,6.176355
65,Sofia,42.697703,23.321736
66,Turin,45.067755,7.682489
